<a href="https://colab.research.google.com/github/aishanikar9/BWSI_Operations_Team/blob/main/OluJ_SegmentationModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q kagglehub
import kagglehub, pathlib, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")
print("downloaded to:", path)

Using Colab cache for faster access to the 'rescuenet' dataset.
downloaded to: /kaggle/input/rescuenet


In [3]:
root = pathlib.Path(path)
org_dir   = list(root.rglob("train-org-img"))[0]
label_dir = list(root.rglob("train-label-img"))[0]
print("images:", len(list(org_dir.glob("*.jpg"))), "| masks:", len(list(label_dir.glob("*.png"))))

images: 3595 | masks: 3595


In [4]:
#0=Unlabeled, 1= Water, 2 = Building w/o Damage
#3 Building with minor damage, 4 Building with Major damage
#5 Building completely destroyed, 6 Vechicle, 7 Clear Road
#8 Blocked Road, 9 Tree, 10 Pool

!pip install -q segmentation-models-pytorch
import segmentation_models_pytorch as smp
import torch, torch.nn as nn, torch.optim as optim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.2 MB/s eta 0:00:00


In [5]:
class_number = 12 #masks actually contain 1 more id than images so this was causing the "TORCH_USE_CUDA_DSA` to enable device-side assertions." error

model = smp.Unet(encoder_name="resnet34",
                 encoder_weights="imagenet",
                 in_channels=3,
                 classes=class_number,
                 )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

In [6]:
path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")
root = pathlib.Path(path)

train_original   = list(root.rglob("train-org-img"))[0] #i misspelled original, say you swr bro.
train_label = list(root.rglob("train-label-img"))[0]
val_original     = list(root.rglob("val-org-img"))[0]
val_label   = list(root.rglob("val-label-img"))[0]
print("train imgs/masks:", len(list(train_original.glob('*.jpg'))), len(list(train_label.glob('*.png'))))
print("val imgs/masks:  ", len(list(val_original.glob('*.jpg'))), len(list(val_label.glob('*.png'))))

Using Colab cache for faster access to the 'rescuenet' dataset.
train imgs/masks: 3595 3595
val imgs/masks:   449 449


In [7]:
loss = nn.CrossEntropyLoss()
dice_loss = smp.losses.DiceLoss(mode="multiclass") # diceloss is used common in image segmentation by focusing on the intersection of predicted mask and the real mask

def criterion(logits, masks):
  return loss(logits, masks) + dice_loss(logits, masks)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, 0.9)

In [8]:
import pathlib, numpy as np
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
from torchvision import tv_tensors #allows masks to be wrapped and treated as labels
from torch.utils.data import DataLoader #forgot to import DataLoader
from pathlib import Path #Path hadn't been imported yet

In [ ]:
#RescueNetSegmentedDataset class, Kento

In [9]:
#build the data sets from org_dir, label_dir, see ResNetModel for reference, Oluj
class RescueNetDataset(Dataset):
    def __init__(self, org_dir, label_dir, transform=None): # added a transform parameter instead of size and train to match with other code
        self.org_dir = pathlib.Path(org_dir)
        self.label_dir = pathlib.Path(label_dir)
        self.transform = transform
        self.images = sorted(f for f in os.listdir(org_dir) if f.endswith(".jpg")) #sorts .jpg to skip unwanted files

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = Path(self.images[idx])
        full_image_path = os.path.join(self.org_dir, self.images[idx]) # prepended to image_dir to allow the file to be findable again
        mask_path = os.path.join(self.label_dir, img_path.stem +"_lab.png") #heard you guys saying os was better

        image =  Image.open(full_image_path).convert("RGB") #converted to RGB channels because it's n image, also opening the full path now
        mask = tv_tensors.Mask(np.array(Image.open(mask_path))) #wraping the mask in a tensor to have transform resize it NEARESt

        if self.transform:
          image, mask = self.transform(image, mask) #applies the transformation to the image and mask and then rebinds both back together
        return image, mask

In [10]:
#IoU or mIOU evaluation function, Aishani
def calc_IOU(pred_arr, label_arr): #since this uses np.sum, I had to convert "preds" and "y" in the next cell
  #multiclass - each class has separate IOU score?
  iou_scores = []
  #union pix
  for c_index in range(class_number):
    #get where they overlap and are correct
    class_gt = label_arr == c_index
    class_pred_area = pred_arr == c_index
    intersect_pix = np.sum((class_gt & class_pred_area))
    intersect_pix = float(intersect_pix)

    #get total area of masks merged

    total_region = np.sum((class_gt | class_pred_area))
    total_region = float(total_region)

    if(total_region == 0):
      continue

    iou_scores.append(intersect_pix/total_region)

  return iou_scores

def calc_mean_IOU(iou_scores): # can merge into another output above
  #iou_scores = [score for score in iou_scores if score != 0]
  return np.mean(iou_scores)


#testing
# import PIL
# import os
# from pathlib import Path
# import matplotlib.pyplot as plt
# c = 0
# p_img_masks = []
# for f in Path(train_label).iterdir():
#   if c<5:
#     img = PIL.Image.open(f)
#     img_arr = np.array(img)
#     p_img_masks.append(img_arr)
#   c+=1

# print(calc_IOU(p_img_masks[4],p_img_masks[1]))
# fix ,ax = plt.subplots(1,2,figsize=(6,6))

# ax[0].imshow(p_img_masks[4])
# ax[1].imshow(p_img_masks[1])

In [11]:
#training, saving checkpoints to drive, Brian
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])

def get_loaders(train_dir, train_mask_dir, val_dir, val_mask_dir, batch_size, train_transform, val_transform, num_workers=4, pin_memory=True):
    train_dataset = RescueNetDataset(org_dir=train_dir, label_dir=train_mask_dir, transform=train_transform) # kept names consistent, image_dir/mask_dir is now org_dir/label_dir
    train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory, shuffle=True)

    val_dataset = RescueNetDataset(org_dir=val_dir, label_dir=val_mask_dir, transform=val_transform) #same thing, keeping names consistent
    val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory)

    return train_loader, val_loader

def check_accuracy(loader, model, device="cuda"):
    num_correct = 0
    num_pixels = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            preds = torch.argmax(model(x), dim=1)
            num_correct += (preds == y).sum()
            num_pixels += torch.numel(preds)

    print(f"Got {num_correct}/{num_pixels} with accuracy {num_correct/num_pixels*100:.2f}")
    calc_iou_var = calc_IOU(preds.cpu().numpy(), y.cpu().numpy()) #np.sum can't read tensors, first switch tensor to live in ram, then use numpy() to convert that tensor into a Numpy array.
    print(f"Calc IoU: {calc_iou_var}")
    print(f"Calc Mean IoU: {calc_mean_IOU(calc_iou_var)}")
    model.train()

In [12]:
#training, saving checkpoints to drive, Brian
import torch
from torchvision.transforms import v2
from torchvision import tv_tensors
from tqdm import tqdm
import os
import pathlib
import torch.nn as nn
import torch.optim as optim

# path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")

#hyperparameters

alpha = 1e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 16
num_epochs = 3
num_workers = 2
image_height =160
image_width = 256 #for a UNet both dimensions need to be divisible by 32
pin_memory = True
load_model = True
# train_img_dir = os.path.join(path, "RescueNet", "train", "train-org-img")
# train_mask_dir = os.path.join(path, "RescueNet", "train", "train-label-img")
# val_img_dir = os.path.join(path, "RescueNet", "val", "val-org-img")
# val_mask_dir = os.path.join(path, "RescueNet", "val", "val-label-img")

def train(loader, model, optimizer, loss_func, scaler):
    loop = tqdm(loader)

    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=device)
        targets = targets.to(device=device)

        with torch.amp.autocast(device_type=device):
            predictions = model(data)
            loss = loss_func(predictions, targets)

        optimizer.zero_grad() #moved the next four out of the autocast block. Autocast should only wrap around the forward pass to compute predictions and loss, everything else should be outside
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loop.set_postfix(loss=loss.item())


def main():
    train_transforms = v2.Compose([
        v2.Resize((image_height, image_width)),
        v2.RandomRotation(degrees=35),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.1),
        v2.ToImage(),
        v2.ToDtype({tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.int64, "others": None}, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), #pretrained encoder needs the ImageNet normalization at the end since that's the incoder I initialized the model with
    ])

    val_transforms = v2.Compose([
        v2.Resize((image_height, image_width)),
        v2.ToImage(),
        v2.ToDtype({tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.int64, "others": None}, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), #val needs the same normalization
    ])

    # model = UNET(in_channels=3, out_channels=3).to(device)
    loss_func = criterion #used criterion loss function, cross entropy is per pixel, dice loss is per overlap, using these loss calculations together is standard for segmentation loss for multi-class problems
    optimizer = optim.Adam(model.parameters(), lr=alpha)

    train_loader, val_loader = get_loaders(train_original, train_label,
                                           val_original, val_label,
                                           batch_size,
                                           train_transforms, val_transforms,
                                           num_workers, pin_memory)

    scaler = torch.amp.GradScaler()

    for epoch in range(num_epochs):
        train(train_loader, model, optimizer, loss_func, scaler)

        checkpoint = {"state_dict": model.state_dict(),
                      "optimizer": optimizer.state_dict()}
        save_checkpoint(checkpoint)

        check_accuracy(val_loader, model, device=device)

    #saving the model this time.
    torch.save({"state_dict": model.state_dict()}, "/content/drive/MyDrive/oluj_unet_best.pth")
    print("saved:", pathlib.Path("/content/drive/MyDrive/oluj_unet_best.pth").exists()) #must print True

if __name__ == "__main__":
  print("Training has begun!")
  main()

Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!
Training has begun!


RecursionError: maximum recursion depth exceeded